# Modelado e inferencia (MiBici) — Fase 2

Este notebook **no entrena ni registra modelos**: solo consume lo que generan los módulos.

1. `python -m mibici.features v3` genera el dataset `mibici_processed_v3`.
2. `python -m mibici.modeling.compare_models` evalúa las líneas base, Ridge y HGB Poisson con
   walk-forward, registra los runs en MLflow y deja al mejor modelo como `champion`.
3. Aquí se carga el dataset v3, se muestra la comparación registrada en MLflow y se predice
   con el `champion` mediante `mibici.modeling.predict`.

Toda la configuración se lee de `params.yaml`.

In [ ]:
import os
import warnings

from loguru import logger

# Evitar rutas locales en las salidas: mensajes de import de mibici, la sugerencia de MLflow
# y el aviso de tqdm (sin ipywidgets)
logger.disable("mibici")
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
import mlflow
import pandas as pd

from mibici.config import MLFLOW_TRACKING_URI, PROCESSED_DATA_DIR, cargar_parametros
from mibici.modeling.predict import cargar_modelo, predecir
from mibici.modeling.validation import METRICAS

logger.enable("mibici")

params = cargar_parametros()

## 1. Dataset v3

In [ ]:
datos = pd.read_parquet(PROCESSED_DATA_DIR / "estacion_hora_v3.parquet")

print(f"Dataset v3: {datos.shape[0]:,} filas x {datos.shape[1]} columnas")
print(f"Periodo: {datos['fecha_hora'].min()} a {datos['fecha_hora'].max()}")
datos.head()

## 2. Comparación de modelos (MLflow)

Métricas promedio de las ventanas walk-forward, tomadas de los runs padre que registra
`compare_models` en el experimento de `params.yaml`. Si un modelo se corrió varias veces se
muestra su run más reciente.

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

runs = mlflow.search_runs(
    experiment_names=[params["mlflow"]["experimento"]],
    order_by=["attributes.start_time DESC"],
)

# Runs padre de modelos: tienen métricas promedio y no tienen padre
columnas_metricas = [f"metrics.{nombre}_media" for nombre in METRICAS]
padres = runs[runs["tags.mlflow.parentRunId"].isna() & runs[columnas_metricas[0]].notna()]

comparacion = (
    padres.drop_duplicates("tags.mlflow.runName")
    .set_index("tags.mlflow.runName")[columnas_metricas]
    .rename_axis("modelo")
    .rename(columns=lambda columna: columna.removeprefix("metrics."))
    .sort_values(f"{next(iter(METRICAS))}_media")
)
comparacion

## 3. Modelo champion

In [ ]:
nombre_modelo = params["mlflow"]["modelo_registrado"]

version = mlflow.MlflowClient().get_model_version_by_alias(nombre_modelo, "champion")

print(f"{nombre_modelo}@champion -> versión {version.version} (run {version.run_id})")
pd.Series(version.tags, name="valor").to_frame()

## 4. Inferencia con `mibici.modeling.predict`

Se predice el último mes de prueba de `params.yaml`. El `champion` se entrenó con todo el
periodo, así que estas métricas son **dentro de muestra**: sirven para comprobar que la carga
y la predicción funcionan, no para comparar modelos (eso es la sección 2).

In [ ]:
modelo = cargar_modelo()

mes = pd.Period(params["validacion"]["mes_prueba_fin"], freq="M")
muestra = datos[datos["fecha_hora"].dt.to_period("M") == mes]

predicciones = predecir(muestra, modelo=modelo)

metricas = {
    nombre: metrica(muestra["salidas_siguiente_hora"], predicciones)
    for nombre, metrica in METRICAS.items()
}
print(f"{mes}: {len(muestra):,} predicciones | {metricas}")

resultado = muestra[["estacion_id", "fecha_hora", "salidas_siguiente_hora"]].copy()
resultado["prediccion"] = predicciones
resultado.head(10)